# Models

In [17]:
import pandas as pd
df = pd.read_csv("../../Merged_Data/merged_flights.csv",low_memory=False)

In [18]:
df

,year,month,day_of_month,day_of_week,fl_date,op_unique_carrier,op_carrier_fl_num,origin_city,origin_state,dest_city,...,IS_Delay,Season,Departure_Hour,Temperature_C,Humidity_pct,Precipitation_mm,Wind_Speed_kmh,Date,Time,Weather_Data_Present
0,2024,1,1,1,2024-01-01,OO,3862.0,Great Falls,Montana,Salt Lake City,...,0,Winter,5,NaN,NaN,NaN,NaN,NaN,NaN,No
1,2024,1,1,1,2024-01-01,B6,148.0,Las Vegas,Nevada,New York,...,1,Winter,17,NaN,NaN,NaN,NaN,NaN,NaN,No
2,2024,1,1,1,2024-01-01,WN,205.0,Las Vegas,Nevada,Portland,...,0,Winter,17,NaN,NaN,NaN,NaN,NaN,NaN,No
3,2024,1,1,1,2024-01-01,WN,1881.0,Las Vegas,Nevada,Indianapolis,...,1,Winter,17,NaN,NaN,NaN,NaN,NaN,NaN,No
4,2024,1,1,1,2024-01-01,WN,2675.0,Las Vegas,Nevada,Atlanta,...,1,Winter,17,NaN,NaN,NaN,NaN,NaN,NaN,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1343438,2024,12,31,2,2024-12-31,WN,3622.0,Columbus,Ohio,Nashville,...,0,Winter,19,NaN,NaN,NaN,NaN,NaN,NaN,No
1343439,2024,12,31,2,2024-12-31,OH,5558.0,Columbus,Ohio,Charlotte,...,0,Winter,18,NaN,NaN,NaN,NaN,NaN,NaN,No
1343440,2024,12,31,2,2024-12-31,WN,4899.0,Columbus,Ohio,Las Vegas,...,0,Winter,17,NaN,NaN,NaN,NaN,NaN,NaN,No
1343441,2024,12,31,2,2024-12-31,WN,3119.0,New Orleans,Louisiana,Burbank,...,1,Winter,13,NaN,NaN,NaN,NaN,NaN,NaN,No


## Baseline model

Predict majority class of no delay

In [24]:
from sklearn.metrics import precision_score, recall_score, f1_score

y_true = df['IS_Delay']
y_pred_baseline = [0] * len(df)

print(df['IS_Delay'].value_counts())
baseline_accuracy = (df['IS_Delay'] == 0).sum() / len(df)
print(f"Accuracy: {baseline_accuracy:.3f}")
print(f"Precision: {precision_score(y_true, y_pred_baseline, zero_division=0):.3f}")
print(f"Recall: {recall_score(y_true, y_pred_baseline, zero_division=0):.3f}")
print(f"F1 score: {f1_score(y_true, y_pred_baseline, zero_division=0):.3f}")

IS_Delay
0    949747
1    393696
Name: count, dtype: int64
Accuracy: 0.707
Precision: 0.000
Recall: 0.000
F1 score: 0.000


## Correlation of target with selected features

In [20]:
from sklearn.preprocessing import LabelEncoder
df_1 = df.copy()

cols = ["month","day_of_month","day_of_week","op_unique_carrier","origin_city","origin_state",
    "dep_time","Season","Departure_Hour","IS_Delay", "Precipitation_mm"]

df_1 = df_1[cols].dropna()

#encoding
for col in ["op_unique_carrier", "origin_city", "origin_state", "Season"]:
    df_1[col] = LabelEncoder().fit_transform(df_1[col])


corr = df_1.corr(numeric_only=True)
print(corr["IS_Delay"].sort_values(ascending=False))

IS_Delay             1.000000
dep_time             0.201703
Departure_Hour       0.159979
day_of_week          0.032353
month                0.029226
origin_state         0.019945
op_unique_carrier    0.010975
Precipitation_mm     0.001083
Season              -0.031399
day_of_month        -0.033025
origin_city         -0.047715
Name: IS_Delay, dtype: float64


# Simple decision tree bagging

In [26]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix


df_1 = df.copy()
cols = ["month","day_of_month","day_of_week", "op_unique_carrier","origin_city","origin_state","Precipitation_mm",
    "dep_time","Season","Departure_Hour","IS_Delay"]
df_1 = df_1[cols].dropna()

# Encode all at once
le = LabelEncoder()
for col in ["op_unique_carrier", "origin_city", "origin_state", "Season"]:
    df_1[col] = le.fit_transform(df_1[col])


X = df_1.drop("IS_Delay", axis=1)
y = df_1["IS_Delay"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
tree = DecisionTreeClassifier()

# Bagging with 5 trees
model_bag = BaggingClassifier(estimator=tree,n_estimators=1)


model_bag.fit(X_train, y_train)
y_pred = model_bag.predict(X_test)
y_proba = model_bag.predict_proba(X_test)[:, 1]

print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred):.3f}")
print(f"Recall: {recall_score(y_test, y_pred):.3f}")
print(f"F1 Score: {f1_score(y_test, y_pred):.3f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.3f}")

print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
print(f"\nConfusion Matrix:")
print(f"TN: {cm[0,0]:5d}  FP: {cm[0,1]:5d}")
print(f"FN: {cm[1,0]:5d}  TP: {cm[1,1]:5d}")

Accuracy: 0.770
Precision: 0.591
Recall: 0.623
F1 Score: 0.606
ROC-AUC: 0.726
              precision    recall  f1-score   support

           0       0.85      0.83      0.84     15563
           1       0.59      0.62      0.61      6205

    accuracy                           0.77     21768
   macro avg       0.72      0.73      0.72     21768
weighted avg       0.77      0.77      0.77     21768


Confusion Matrix:
TN: 12886  FP:  2677
FN:  2340  TP:  3865


## Hyperparameter tuning for bagging

No bagging (n=1): 0.777

Bagging with n = 2: 0.8274

Bagging with n = 3: 0.8137

Bagging with n = 5: 0.8242

# KNN

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix



df_1 = df.copy()
df_1 = df_1.sample(2000,random_state=811)
cols = ["month","day_of_month","day_of_week","op_unique_carrier","origin_city","origin_state","dep_time","Season","Departure_Hour","IS_Delay", "Precipitation_mm"]
df_1 = df_1[cols].dropna()
df_1 = pd.get_dummies(df_1, columns=["op_unique_carrier","origin_city","origin_state","Season"], drop_first=True)

X = df_1.drop("IS_Delay", axis=1)
y = df_1["IS_Delay"]


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
# Scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
knn = KNeighborsClassifier()
param_grid = {"n_neighbors": [5, 10, 15, 20],"weights": ["uniform", "distance"],
    "metric": ["euclidean", "manhattan"]}


grid = GridSearchCV(knn, param_grid, cv=5, return_train_score=True)
grid.fit(X_train, y_train)
results = pd.DataFrame(grid.cv_results_)
results = results[["param_n_neighbors","param_weights","param_metric","mean_test_score"]]
results = results.sort_values(by="mean_test_score", ascending=False)

print(results)

    param_n_neighbors param_weights param_metric  mean_test_score
14                 20       uniform    manhattan         0.691250
6                  20       uniform    euclidean         0.690625
10                 10       uniform    manhattan         0.686250
2                  10       uniform    euclidean         0.686250
12                 15       uniform    manhattan         0.683125
4                  15       uniform    euclidean         0.676250
7                  20      distance    euclidean         0.672500
15                 20      distance    manhattan         0.672500
13                 15      distance    manhattan         0.669375
5                  15      distance    euclidean         0.665625
3                  10      distance    euclidean         0.661250
11                 10      distance    manhattan         0.651250
0                   5       uniform    euclidean         0.650000
8                   5       uniform    manhattan         0.645625
1         

## Analysis of best KNN

In [23]:
best_knn = grid.best_estimator_
y_pred = best_knn.predict(X_test)
y_proba = best_knn.predict_proba(X_test)[:, 1]

print(f"Accuracy:  {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred):.3f}")
print(f"Recall:  {recall_score(y_test, y_pred):.3f}")
print(f"F1Score: {f1_score(y_test, y_pred):.3f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.3f}")

print(classification_report(y_test, y_pred))
cm = confusion_matrix(y_test, y_pred)
print(f"\nConfusion matrix:")
print(f"TN: {cm[0,0]:5d}  FP: {cm[0,1]:5d}")
print(f"FN: {cm[1,0]:5d}  TP: {cm[1,1]:5d}")

Accuracy:  0.698
Precision: 0.316
Recall:  0.053
F1Score: 0.090
ROC-AUC: 0.550
              precision    recall  f1-score   support

           0       0.72      0.95      0.82       286
           1       0.32      0.05      0.09       114

    accuracy                           0.70       400
   macro avg       0.52      0.50      0.45       400
weighted avg       0.60      0.70      0.61       400


Confusion matrix:
TN:   273  FP:    13
FN:   108  TP:     6


## Hyperparameter tuning results for KNN

Results: 
    param_n_neighbors param_weights param_metric  mean_test_score

**4                  15       uniform    euclidean         0.696875   - best model**

6                  20       uniform    euclidean         0.693750   

14                 20       uniform    manhattan         0.693750   

2                  10       uniform    euclidean         0.691250   

12                 15       uniform    manhattan         0.691250   

10                 10       uniform    manhattan         0.690625   

15                 20      distance    manhattan         0.680625   

7                  20      distance    euclidean         0.678125   

5                  15      distance    euclidean         0.676250   

13                 15      distance    manhattan         0.672500   

3                  10      distance    euclidean         0.661250   

0                   5       uniform    euclidean         0.661250   

8                   5       uniform    manhattan         0.659375   

11                 10      distance    manhattan         0.656875   

1                   5      distance    euclidean         0.646875   

9                   5      distance    manhattan         0.645625   